### 1. Setup & Load Tables

In [0]:
from pyspark.sql import functions as F

# Load tables
fact_flights = spark.read.table("Skylens_macro.team1.fact_flights")
fact_daily   = spark.read.table("Skylens_macro.team1.fact_daily_route_summary")

dim_airline  = spark.read.table("Skylens_macro.team1.dim_airline")
dim_origin   = spark.read.table("Skylens_macro.team1.dim_origin_airport")
dim_dest     = spark.read.table("Skylens_macro.team1.dim_destination_airport")
dim_date     = spark.read.table("Skylens_macro.team1.dim_date")
dim_aircraft = spark.read.table("Skylens_macro.team1.dim_aircraft")
dim_cancel   = spark.read.table("Skylens_macro.team1.dim_cancellation_reason")

errors = []

### 2. Null Checks on Critical Keys

In [0]:
null_checks = {
    "date_key": fact_flights.filter(F.col("date_key").isNull()).count(),
    "airline_key": fact_flights.filter(F.col("airline_key").isNull()).count(),
    "origin_airport_key": fact_flights.filter(F.col("origin_airport_key").isNull()).count(),
    "dest_airport_key": fact_flights.filter(F.col("dest_airport_key").isNull()).count(),
    "aircraft_key": fact_flights.filter(F.col("aircraft_key").isNull()).count(),
}

for col, cnt in null_checks.items():
    if cnt > 0:
        errors.append(f"NULL values in {col}: {cnt}")

### 3. Referential Integrity Checks

In [0]:
# airline_key
invalid_airline = fact_flights.join(dim_airline, "airline_key", "left_anti").count()
if invalid_airline > 0:
    errors.append(f"Invalid airline_key: {invalid_airline}")

# origin airport
invalid_origin = fact_flights.join(dim_origin, fact_flights.origin_airport_key == dim_origin.airport_key, "left_anti").count()
if invalid_origin > 0:
    errors.append(f"Invalid origin_airport_key: {invalid_origin}")

# destination airport
invalid_dest = fact_flights.join(dim_dest, fact_flights.dest_airport_key == dim_dest.airport_key, "left_anti").count()
if invalid_dest > 0:
    errors.append(f"Invalid dest_airport_key: {invalid_dest}")

# date_key
invalid_date = fact_flights.join(dim_date, "date_key", "left_anti").count()
if invalid_date > 0:
    errors.append(f"Invalid date_key: {invalid_date}")

### 4. Duplicate Check (Grain Validation)

In [0]:
dup_count = fact_flights.groupBy(
    "date_key", "airline_key", "flight_number", "origin_airport_key"
).count().filter("count > 1").count()

if dup_count > 0:
    errors.append(f"Duplicate records found in fact_flights: {dup_count}")

### 5. Metric Validation (Negative / Invalid Values)

In [0]:
invalid_metrics = fact_flights.filter(
    (F.col("distance_miles") < 0) |
    (F.col("air_time_min") < 0) |
    (F.col("elapsed_time_min") < 0)
).count()

if invalid_metrics > 0:
    errors.append(f"Negative values in metrics: {invalid_metrics}")

### 6. Business Logic Checks

In [0]:
# Cancelled flights should not have delays
invalid_cancel_logic = fact_flights.filter(
    (F.col("is_cancelled") == True) &
    (F.col("arrival_delay_min").isNotNull())
).count()

if invalid_cancel_logic > 0:
    errors.append(f"Cancelled flights have delay values: {invalid_cancel_logic}")

### 7. Percentage Validity (Aggregated Tables)

In [0]:
gold_airline = spark.read.table("Skylens_macro.team1.gold_airline_performance")

invalid_pct = gold_airline.filter(
    (F.col("on_time_pct") < 0) | (F.col("on_time_pct") > 100)
).count()

if invalid_pct > 0:
    errors.append(f"Invalid on_time_pct values: {invalid_pct}")

### 8. Aggregation Consistency Check

In [0]:
silver_count = spark.read.table("workspace.default.silver_flights").count()
gold_total = gold_airline.agg(F.sum("total_flights")).collect()[0][0]

if gold_total > silver_count:
    errors.append("Gold total flights exceed silver count")

### 9. Data Freshness Check

In [0]:
latest_date = fact_flights.agg(F.max("date_key")).collect()[0][0]

if latest_date is None:
    errors.append("No data found in fact_flights")

### 10. Fact Daily Table Validation

In [0]:
invalid_daily = fact_daily.filter(
    (F.col("on_time_pct") < 0) | (F.col("on_time_pct") > 100)
).count()

if invalid_daily > 0:
    errors.append(f"Invalid on_time_pct in fact_daily: {invalid_daily}")

### 11. FINAL ASSERT (FAIL PIPELINE)

In [0]:
if len(errors) > 0:
    print("GOLD DATA QUALITY CHECK FAILED\n")
    for err in errors:
        print(err)
    raise Exception("Gold Layer Data Quality Failed")

else:
    print("GOLD DATA QUALITY CHECK PASSED")